# GSM8K Task

In [1]:
import os
import re
import json
from datasets import load_dataset
from datasets import get_dataset_config_names
from datasets import get_dataset_split_names

/home/user/projects/my-nanochat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def print_colored(convo, limit=float('inf')):
    for i, message in enumerate(convo['messages']):
        if i >= limit:
            print(f"\033[31m... {len(convo['messages']) - limit} more messages ...\033[0m")
            break
        role = message['role']
        content = message['content']
        if role == 'system':
            print(f"\033[33m{content}\033[0m")  # yellow
        elif role == 'assistant':
            if isinstance(content, str):
                print(f"\033[34m{content}\033[0m")  # blue
            elif isinstance(content, list):
                for part in content:
                    if part['type'] == 'text':
                        print(f"\033[34m{part['text']}\033[0m", end='')  # blue
                    elif part['type'] == 'python':
                        print(f"\033[36m{part['text']}\033[0m", end='')  # cyan
                    elif part['type'] == 'python_output':
                        print(f"\033[35m{part['text']}\033[0m", end='')  # magenta
                    else:
                        print(f"\033[31m{part}\033[0m", end='')  # red for unknown part types
                print()  # newline after the assistant message
            else:
                # red
                print(f"\033[31m{content}\033[0m")  # red
                
        elif role == 'user':
            print(f"\033[32m{content}\033[0m")  # green
        else:
            print(f"\033[31m{content}\033[0m")  # red


In [3]:
subsets = get_dataset_config_names("openai/gsm8k")
print(f"Subsets: {subsets}")


splits = get_dataset_split_names("openai/gsm8k", "main")
print(f"Splits: {splits}")

Subsets: ['main', 'socratic']
Splits: ['train', 'test']


In [4]:
ds = load_dataset("openai/gsm8k", "main", split="train")

In [5]:
example = ds[0]

In [6]:
example

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [7]:
question = example['question']
answer = example['answer']  # may contain python tool call in '<<2+3=5>>' format

answer_parts = re.split(r'(<<[^>]+>>)', answer)
assistant_parts = []
for part in answer_parts:
    if part.startswith('<<') and part.endswith('>>'):
        expr_and_maybe_result = part[2:-2]  # remove << and >>
        if '=' in expr_and_maybe_result:
            expr, result = expr_and_maybe_result.split('=', 1)
        else:
            expr, result = expr_and_maybe_result, ""
        assistant_parts.append({"type": "python", "text": expr})
        assistant_parts.append({"type": "python_output", "text": result})
    else:
        assistant_parts.append({"type": "text", "text": part})
convo = {
    "messages": [
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_parts},
    ]
}

In [8]:
convo

{'messages': [{'role': 'user',
   'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'},
  {'role': 'assistant',
   'content': [{'type': 'text', 'text': 'Natalia sold 48/2 = '},
    {'type': 'python', 'text': '48/2'},
    {'type': 'python_output', 'text': '24'},
    {'type': 'text', 'text': '24 clips in May.\nNatalia sold 48+24 = '},
    {'type': 'python', 'text': '48+24'},
    {'type': 'python_output', 'text': '72'},
    {'type': 'text',
     'text': '72 clips altogether in April and May.\n#### 72'}]}]}

In [9]:
print_colored(convo)

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Natalia sold 48/2 = 48/22424 clips in May.
Natalia sold 48+24 = 48+247272 clips altogether in April and May.
#### 72


In [21]:
class TaskGSM8K:
    def __init__(self, subset, split, stop=None):
        self.dataset = load_dataset("openai/gsm8k", subset, split=split)
        print(stop)
        self.length = stop if stop is not None else len(self.dataset)
        print(self.length, len(self.dataset))
    
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError(idx)
        example = self.dataset[idx]
        
        question = example['question']
        answer = example['answer']  # may contain python tool call in '<<2+3=5>>' format

        answer_parts = re.split(r'(<<[^>]+>>)', answer)
        assistant_parts = []
        for part in answer_parts:
            if part.startswith('<<') and part.endswith('>>'):
                expr_and_maybe_result = part[2:-2]  # remove << and >>
                if '=' in expr_and_maybe_result:
                    expr, result = expr_and_maybe_result.split('=', 1)
                else:
                    expr, result = expr_and_maybe_result, ""
                assistant_parts.append({"type": "python", "text": expr})
                assistant_parts.append({"type": "python_output", "text": result})
            else:
                assistant_parts.append({"type": "text", "text": part})
        convo = {
            "messages": [
                {"role": "user", "content": question},
                {"role": "assistant", "content": assistant_parts},
            ]
        }

        return convo

In [22]:
def check_schema(convo):
    assert isinstance(convo, dict)
    assert convo.keys() == {'messages'}
    assert isinstance(convo['messages'], list)
    for message in convo['messages']:
        assert isinstance(message, dict)
        assert message.keys() == {'role', 'content'}
        assert message['role'] in {'system', 'assistant', 'user'}
        if isinstance(message['content'], str):
            assert isinstance(message['content'], str)
        elif isinstance(message['content'], list):
            for part in message['content']:
                assert isinstance(part, dict)
                assert part.keys() == {'type', 'text'}
                assert part['type'] in {'text', 'python', 'python_output'}
                assert isinstance(part['text'], str)
        else:
            assert False, f"Invalid content type: {type(message['content'])}"
        assert len(message['content']) > 0

In [23]:
# Checked ok
task = TaskGSM8K("main", "train")
for i, e in enumerate(task):
    check_schema(e)
    if i % 10_000 == 0:
        print(f"Checked {i} / {len(task)} examples...")



None
7473 7473
Checked 0 / 7473 examples...


In [24]:
# Checked ok
task = TaskGSM8K("main", "test")
for i, e in enumerate(task):
    check_schema(e)
    if i % 10_000 == 0:
        print(f"Checked {i} / {len(task)} examples...")

None
1319 1319
Checked 0 / 1319 examples...
